In [14]:
import sys
from pathlib import Path
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)


import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from src.utils import get_features_data_path
from sklearn.preprocessing import StandardScaler
import joblib
from src.optimization import tune_lasso_optuna, tune_xgb_optuna
from src.data_processing import save_processed_data


In [15]:
data_path = get_features_data_path()
print(f"Szukam pliku: {data_path}")
print(f"Plik istnieje: {data_path.exists()}")

df = pd.read_csv(data_path, parse_dates=['timestamp'])
print(f"✓ Wczytano: {df.shape}")

Szukam pliku: C:\Users\aiszk\PycharmProjects\PythonProject\data\processed\features_data_encoded.csv
Plik istnieje: True
✓ Wczytano: (3931227, 72)


In [16]:
# 1. Sortowanie czasowe
if 'timestamp' in df.columns:
    df = df.sort_values('timestamp').reset_index(drop=True)

# 2. Usuń kolumny niebędące cechami
drop_cols = ['timestamp', 'meter_id', 'campus_id', 'consumption',
             'date_dt', 'id', 'id_building', 'id_campus']
drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols)
y = df['consumption']

# 3. Usuń kolumny kategoryczne (strings)
categorical_cols = X.select_dtypes(include=['object']).columns
X = X.drop(columns=categorical_cols)

# 4. Uzupełnij braki średnią (po kolumnach)
X = X.fillna(X.mean(numeric_only=True))

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# 5. Split 80/20 – kolejność czasową
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


X shape: (3931227, 70)
y shape: (3931227,)
Train: (3144981, 70), Test: (786246, 70)


In [17]:
joblib.dump(y_test, "../data/processed/y_test_fs_optuna.joblib")
print(y_test.shape)

(786246,)


### Feature Selection

In [4]:

# 1. Skalowanie train setu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 2. Trening modelu  Lasso do feature selection
lasso_fs = Lasso(alpha=0.1, max_iter=10000, random_state=42)
lasso_fs.fit(X_train_scaled, y_train)

# 3. Współczynniki jako Series
coefs = pd.Series(lasso_fs.coef_, index=X_train.columns)

# 4. Wybrane cechy (np. wszystkie z |coef| > 1e-5)
selected_features = coefs[coefs.abs() > 1e-5].index.tolist()

print(f"Liczba wybranych cech: {len(selected_features)}")
print(selected_features[:20])  # podgląd



Liczba wybranych cech: 17
['apparent_temperature', 'relative_humidity', 'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'is_morning', 'is_evening', 'consumption_lag_168h', 'consumption_rolling_std_3h', 'consumption_per_sqm', 'consumption_per_person', 'pca_1', 'pca_2', 'pca_3', 'pca_5', 'built_year_1999.0']


In [13]:
# Zredukowane macierze cech
X_train_fs = X_train[selected_features].copy()
X_test_fs = X_test[selected_features].copy()

print("X_train_fs:", X_train_fs.shape)
print("X_test_fs:", X_test_fs.shape)

save_processed_data(X_test_fs, "X_test_fs.csv")

NameError: name 'selected_features' is not defined

## Optuna

### Lasso

In [6]:
# Optuna dla Lasso na X_train_fs / X_test_fs
study_lasso, best_lasso, scaler_lasso = tune_lasso_optuna(
    X_train_fs=X_train_fs,
    X_test_fs=X_test_fs,
    y_train=y_train,
    y_test=y_test,
    n_trials=30,
    save_path_model=str(models_dir / "best_lasso_optuna.joblib"),
    save_path_scaler=str(models_dir / "best_lasso_scaler.joblib"),
    save_study_path=str(models_dir / "lasso_optuna_study.joblib"),
)

print("=== BEST LASSO PARAMS ===")
print(study_lasso.best_params)
print("BEST LASSO RMSE:", study_lasso.best_value)


[I 2025-11-27 20:44:04,760] A new study created in memory with name: lasso_optuna
[I 2025-11-27 20:44:12,341] Trial 0 finished with value: 2.9214290274767682 and parameters: {'alpha': 0.21075682372761895, 'max_iter': 19532}. Best is trial 0 with value: 2.9214290274767682.
[I 2025-11-27 20:44:23,994] Trial 1 finished with value: 2.871013445983687 and parameters: {'alpha': 0.00012783847868753733, 'max_iter': 19069}. Best is trial 1 with value: 2.871013445983687.
[I 2025-11-27 20:44:33,627] Trial 2 finished with value: 2.8709226084110004 and parameters: {'alpha': 0.0011177748879199396, 'max_iter': 14675}. Best is trial 2 with value: 2.8709226084110004.
[I 2025-11-27 20:44:41,987] Trial 3 finished with value: 2.8712734582644237 and parameters: {'alpha': 0.0196981547839931, 'max_iter': 5387}. Best is trial 2 with value: 2.8709226084110004.
[I 2025-11-27 20:44:52,126] Trial 4 finished with value: 2.8709848317625632 and parameters: {'alpha': 0.00042011779954474054, 'max_iter': 12557}. Best is

=== BEST LASSO PARAMS ===
{'alpha': 0.004877118385512125, 'max_iter': 11265}
BEST LASSO RMSE: 2.8707390303813054


### XGBOOST

In [7]:
study_xgb, best_xgb = tune_xgb_optuna(
    X_train_fs=X_train_fs,
    X_test_fs=X_test_fs,
    y_train=y_train,
    y_test=y_test,
    n_trials=30,
    save_path_model=str(models_dir / "best_xgb_optuna.joblib"),
    save_study_path=str(models_dir / "xgb_optuna_study.joblib"),
)

print("=== BEST XGB PARAMS ===")
print(study_xgb.best_params)
print("BEST XGB RMSE:", study_xgb.best_value)


[I 2025-11-27 20:48:33,456] A new study created in memory with name: xgboost_optuna
[I 2025-11-27 20:51:13,911] Trial 0 finished with value: 1.6438778895895885 and parameters: {'max_depth': 8, 'learning_rate': 0.01928311957263536, 'subsample': 0.7923447857700112, 'colsample_bytree': 0.608475051788441, 'min_child_weight': 6.686536474119919, 'gamma': 4.5122889485897435, 'n_estimators': 592, 'reg_lambda': 3.568246664908736, 'reg_alpha': 0.04835265928201375}. Best is trial 0 with value: 1.6438778895895885.
[I 2025-11-27 20:52:29,181] Trial 1 finished with value: 2.0433269209545726 and parameters: {'max_depth': 5, 'learning_rate': 0.03451721302414658, 'subsample': 0.6757908109061833, 'colsample_bytree': 0.7931554975330392, 'min_child_weight': 7.333974727194163, 'gamma': 3.5144381642903104, 'n_estimators': 377, 'reg_lambda': 1.1516762305107648, 'reg_alpha': 1.2212142799694858}. Best is trial 0 with value: 1.6438778895895885.
[I 2025-11-27 20:54:34,367] Trial 2 finished with value: 1.29023272

=== BEST XGB PARAMS ===
{'max_depth': 9, 'learning_rate': 0.13562163261848628, 'subsample': 0.997428856641209, 'colsample_bytree': 0.8639904977861322, 'min_child_weight': 9.723898430999915, 'gamma': 2.1000681750189085, 'n_estimators': 457, 'reg_lambda': 0.004516617175164232, 'reg_alpha': 0.0012812135579653033}
BEST XGB RMSE: 1.2548503264749906


In [12]:
import joblib
from sklearn.metrics import mean_squared_error, r2_score

X_test_fs = pd.read_csv("../data/processed/X_test_fs.csv")

print("X_train_fs:", X_train_fs.shape)
print("X_test_fs: ", X_test_fs.shape)
print("y_test:    ", y_test.shape)

# użyj dokładnie tego best_xgb, który wypluła Optuna (tu go masz w zmiennej best_xgb)
y_pred_xgb_05 = best_xgb.predict(X_test_fs)

print("RMSE 05:", mean_squared_error(y_test, y_pred_xgb_05, squared=False))
print("R²   05:", r2_score(y_test, y_pred_xgb_05))

joblib.dump(best_xgb, "../models/best_xgb_optuna_fresh.joblib")


NameError: name 'X_train_fs' is not defined